# SATCP: clasificador de fuego / humo / normal para ESP32-S3

Entrena un **MobileNetV2 (alpha 0.35, 96x96)** con aprendizaje por transferencia y lo exporta a **TFLite int8** como arreglo de C para TensorFlow Lite Micro.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU (T4)**.

Estructura en Google Drive:
```
MyDrive/SATCP/
  dataset/fuego/    dataset/humo/    dataset/normal/
  modelo/           <- aquí queda el modelo exportado
```

**Criterio de clases:**
- `fuego`: **cualquier llama**, tanto incendios como la **llama de la estufa de gas** (azul o naranja, con y sin olla encima).
- `humo`: humo de algo quemándose.
- `normal`: estufa apagada, cocina sin llamas.

**Importante:** los datasets públicos son casi todos de incendios con llamas naranjas. Para que el modelo reconozca la **llama azul del gas**, agrega a `fuego` fotos de tus hornillas encendidas tomadas por el ESP32 (`/sdcard/dataset/`).

In [ ]:
# 1. Configuración
TAM_IMAGEN = 96          # Debe coincidir con el firmware del ESP32
ALPHA = 0.35             # Ancho de MobileNetV2 (0.35 = el más liviano con pesos de ImageNet)
CLASES = ['fuego', 'humo', 'normal']   # El orden define los índices de salida en el ESP32

# Con Drive el dataset y el modelo se conservan entre sesiones.
# Sin Drive todo vive en /content y se borra al cerrar Colab: descarga el .zip del paso 11.
USAR_DRIVE = True
CARPETA = '/content/drive/MyDrive/SATCP' if USAR_DRIVE else '/content/SATCP'
DATASET = CARPETA + '/dataset'
SALIDA = CARPETA + '/modelo'
SEMILLA = 42
LOTE = 32

In [ ]:
# 2. Montar Google Drive (si se usa) y crear carpetas
import os
if USAR_DRIVE:
    from google.colab import drive
    # Si falla con 'credential propagation was unsuccessful': acepta TODOS los permisos
    # con la misma cuenta de Colab, o pon USAR_DRIVE = False en el paso 1
    drive.mount('/content/drive', force_remount=True)
for clase in CLASES:
    os.makedirs(os.path.join(DATASET, clase), exist_ok=True)
os.makedirs(SALIDA, exist_ok=True)
print('Listo:', DATASET)

## 3. Descargar datasets públicos de Kaggle

1. Ejecuta la celda de inicio de sesión (te pedirá tu usuario y token de Kaggle: kaggle.com → Settings → API → Create New Token).
2. Usa `explorar('usuario/dataset')` para ver las carpetas reales de cada dataset.
3. Completa `FUENTES` indicando qué carpeta va a qué clase y ejecuta la copia.

Los nombres sugeridos en `FUENTES` son ejemplos: **verifica en kaggle.com que existan** y ajusta las carpetas a lo que muestre `explorar()`. Busca en Kaggle: *fire dataset*, *fire and smoke classification*, *smoke detection*, *gas stove flame*, *gas burner*.

In [ ]:
!pip -q install kagglehub
import kagglehub
kagglehub.login()

In [ ]:
import glob, random, shutil

EXTENSIONES = ('.jpg', '.jpeg', '.png', '.bmp')
MAX_POR_CARPETA = 1500   # Copiar miles de archivos a Drive es lento; 1500 por carpeta alcanza

def explorar(slug, profundidad=3):
    raiz = kagglehub.dataset_download(slug)
    print('Descargado en', raiz)
    for actual, carpetas, archivos in os.walk(raiz):
        nivel = actual[len(raiz):].count(os.sep)
        if nivel > profundidad:
            continue
        imagenes = [a for a in archivos if a.lower().endswith(EXTENSIONES)]
        print('  ' * nivel + os.path.relpath(actual, raiz), '-', len(imagenes), 'imágenes')

def copiar_fuente(slug, mapeo):
    raiz = kagglehub.dataset_download(slug)
    prefijo = slug.replace('/', '_')
    for origen, clase in mapeo.items():
        archivos = sorted(p for p in glob.glob(os.path.join(raiz, origen, '**', '*'), recursive=True)
                          if p.lower().endswith(EXTENSIONES))
        if not archivos:
            print(f'AVISO: {slug}/{origen} no tiene imágenes; revisa la ruta con explorar()')
            continue
        # Muestra aleatoria pero reproducible, para no quedarse solo con las primeras del dataset
        random.Random(SEMILLA).shuffle(archivos)
        archivos = archivos[:MAX_POR_CARPETA]
        carpeta_origen = origen.replace('/', '_')
        for i, ruta in enumerate(archivos):
            extension = os.path.splitext(ruta)[1].lower()
            destino = os.path.join(DATASET, clase, f'{prefijo}_{carpeta_origen}_{i:06d}{extension}')
            if not os.path.exists(destino):
                shutil.copy(ruta, destino)
        print(f'{slug}/{origen}: {len(archivos)} imágenes -> {clase}')

# Primero revisa las carpetas reales:
# explorar('antoninomaio/smoke-and-fire-detection')
# explorar('jayaprakashpondy/fire-and-smoke')

In [ ]:
# (slug de Kaggle, {carpeta dentro del dataset: clase destino})
# Las rutas son aproximadas: ajústalas con lo que muestre explorar()
FUENTES = [
    # Ya copiado (755 fuego, 244 normal). No lo repitas: los nombres cambiaron y se duplicaría
    # ('phylake1337/fire-dataset', {
    #     'fire_dataset/fire_images': 'fuego',
    #     'fire_dataset/non_fire_images': 'normal',
    # }),
    ('antoninomaio/smoke-and-fire-detection', {
        'Fire': 'fuego',
        'Non_fire': 'normal',
        'Smoke': 'humo',
    }),
    ('jayaprakashpondy/fire-and-smoke', {
        'train/Fire': 'fuego',
        'train/Normal': 'normal',
        'train/Smoke': 'humo',
    }),
]
for slug, mapeo in FUENTES:
    copiar_fuente(slug, mapeo)

In [ ]:
# 4. Revisar el dataset y eliminar imágenes corruptas
import tensorflow as tf

conteo = {}
for clase in CLASES:
    carpeta = os.path.join(DATASET, clase)
    validas = 0
    for nombre in os.listdir(carpeta):
        ruta = os.path.join(carpeta, nombre)
        try:
            tf.io.decode_image(tf.io.read_file(ruta), channels=3, expand_animations=False)
            validas += 1
        except Exception:
            print('Eliminando imagen corrupta:', ruta)
            os.remove(ruta)
    conteo[clase] = validas
print(conteo)
assert all(n >= 50 for n in conteo.values()), 'Cada clase necesita al menos 50 imágenes (idealmente 500+)'

In [ ]:
# 5. Cargar imágenes. Se redimensionan sin conservar proporción, igual que en el ESP32
entrenamiento, validacion = tf.keras.utils.image_dataset_from_directory(
    DATASET, labels='inferred', label_mode='int', class_names=CLASES,
    validation_split=0.2, subset='both', seed=SEMILLA,
    image_size=(TAM_IMAGEN, TAM_IMAGEN), batch_size=LOTE)

# Pesos por clase: compensan que haya muchas más imágenes de una clase que de otra
total = sum(conteo.values())
pesos_clase = {i: total / (len(CLASES) * conteo[c]) for i, c in enumerate(CLASES)}
print('Pesos por clase:', pesos_clase)

aumento = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.25, value_range=(0, 255)),
    tf.keras.layers.RandomContrast(0.25),
])
AUTOTUNE = tf.data.AUTOTUNE
entrenamiento_aumentado = (entrenamiento
    .map(lambda x, y: (aumento(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE))
validacion = validacion.cache().prefetch(AUTOTUNE)

In [ ]:
# 6. Modelo: el escalado a [-1, 1] va dentro, así el ESP32 entrega los píxeles tal cual
base = tf.keras.applications.MobileNetV2(
    input_shape=(TAM_IMAGEN, TAM_IMAGEN, 3), alpha=ALPHA, include_top=False, weights='imagenet')
base.trainable = False

entrada = tf.keras.Input(shape=(TAM_IMAGEN, TAM_IMAGEN, 3))
x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(entrada)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
salida = tf.keras.layers.Dense(len(CLASES), activation='softmax')(x)
modelo = tf.keras.Model(entrada, salida)

parada = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

# Fase 1: solo la capa nueva
modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelo.fit(entrenamiento_aumentado, validation_data=validacion, epochs=15,
           class_weight=pesos_clase, callbacks=[parada])

# Fase 2: ajuste fino de las últimas capas de MobileNetV2
base.trainable = True
for capa in base.layers[:-30]:
    capa.trainable = False
modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
               loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelo.fit(entrenamiento_aumentado, validation_data=validacion, epochs=15,
           class_weight=pesos_clase, callbacks=[parada])

In [ ]:
# 7. Evaluación. Lo más importante: que casi no se escape fuego ni humo (recall alto)
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

reales, predichas = [], []
for imagenes, etiquetas in validacion:
    reales.extend(etiquetas.numpy())
    predichas.extend(np.argmax(modelo.predict(imagenes, verbose=0), axis=1))
print(classification_report(reales, predichas, target_names=CLASES, digits=3))
print('Matriz de confusión (filas = real, columnas = predicho):')
print(CLASES)
print(confusion_matrix(reales, predichas))

In [ ]:
# 8. Convertir a TFLite int8 (entrada y salida int8, como espera TFLite Micro)
def datos_representativos():
    for imagenes, _ in entrenamiento.take(100):
        for i in range(imagenes.shape[0]):
            yield [tf.cast(imagenes[i:i + 1], tf.float32)]

convertidor = tf.lite.TFLiteConverter.from_keras_model(modelo)
convertidor.optimizations = [tf.lite.Optimize.DEFAULT]
convertidor.representative_dataset = datos_representativos
convertidor.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
convertidor.inference_input_type = tf.int8
convertidor.inference_output_type = tf.int8
modelo_tflite = convertidor.convert()

with open(os.path.join(SALIDA, 'modelo_fuego.tflite'), 'wb') as f:
    f.write(modelo_tflite)
print(f'Tamaño del modelo: {len(modelo_tflite) / 1024:.1f} KB')

# Operaciones que debe registrar el firmware en TFLite Micro
tf.lite.experimental.Analyzer.analyze(model_content=modelo_tflite)

In [ ]:
# 9. Verificar que el modelo int8 conserva la precisión
interprete = tf.lite.Interpreter(model_content=modelo_tflite)
interprete.allocate_tensors()
entrada_info = interprete.get_input_details()[0]
salida_info = interprete.get_output_details()[0]
escala_in, cero_in = entrada_info['quantization']
escala_out, cero_out = salida_info['quantization']
print('Entrada:', entrada_info['shape'], 'escala', escala_in, 'cero', cero_in)
print('Salida :', salida_info['shape'], 'escala', escala_out, 'cero', cero_out)

aciertos = total_val = 0
for imagenes, etiquetas in validacion.unbatch().batch(1):
    q = np.round(imagenes.numpy() / escala_in + cero_in)
    interprete.set_tensor(entrada_info['index'], np.clip(q, -128, 127).astype(np.int8))
    interprete.invoke()
    aciertos += int(np.argmax(interprete.get_tensor(salida_info['index'])[0]) == int(etiquetas[0]))
    total_val += 1
print(f'Precisión TFLite int8: {aciertos / total_val:.3f}')

In [ ]:
# 10. Generar los archivos para el firmware (copiar a components/vision/ del proyecto)
import json

with open(os.path.join(SALIDA, 'modelo_fuego.cc'), 'w') as f:
    f.write('// Generado por entrenar_clasificador_fuego.ipynb\n')
    f.write('#include "modelo_fuego.h"\n\n')
    f.write('alignas(16) const unsigned char modelo_fuego[] = {\n')
    for i in range(0, len(modelo_tflite), 12):
        f.write('    ' + ', '.join(f'0x{b:02x}' for b in modelo_tflite[i:i + 12]) + ',\n')
    f.write('};\n')
    f.write(f'const unsigned int modelo_fuego_len = {len(modelo_tflite)};\n')

parametros = {
    'clases': CLASES, 'tam_imagen': TAM_IMAGEN,
    'escala_entrada': float(escala_in), 'cero_entrada': int(cero_in),
    'escala_salida': float(escala_out), 'cero_salida': int(cero_out),
    'bytes': len(modelo_tflite),
}
with open(os.path.join(SALIDA, 'parametros.json'), 'w') as f:
    json.dump(parametros, f, indent=2)
print(json.dumps(parametros, indent=2))
print('Archivos en', SALIDA, ': modelo_fuego.cc, modelo_fuego.tflite, parametros.json')

In [ ]:
# 11. Descargar el modelo a tu computadora (imprescindible si USAR_DRIVE = False)
import shutil
from google.colab import files
archivo_zip = shutil.make_archive('/content/modelo_satcp', 'zip', SALIDA)
files.download(archivo_zip)

## Reentrenar con las fotos del ESP32

1. Copia de la SD las carpetas `dataset/sNNNN/` (capturas periódicas) y `evidencia/sNNNN/` (fotos de la fase visual).
2. Revísalas y muévelas a `MyDrive/SATCP/dataset/fuego`, `humo` o `normal` según lo que se ve.
   - Hornilla encendida (con o sin olla) o cualquier llama → `fuego`.
   - Estufa apagada, ollas sin llama debajo → `normal`.
   - Las fotos de `evidencia` que el sistema clasificó mal son las más valiosas.
3. Vuelve a ejecutar desde el paso 4. El firmware solo necesita el nuevo `modelo_fuego.cc`.